# N043 · 二分答案与可行性判定

**目标：** 证明谓词单调后在答案空间搜索。

**先修：** N042, N016。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最小可行/最大可行；上下界；整数向上取整；判定器成本。

## 从问题到算法

把求最优值转换为判断给定答案是否可行，再证明可行性随答案单调。吃香蕉速度越快，所需小时数不增加；船容量越大，所需天数不增加。边界必须覆盖一个可行答案；整数天数用整除向上取整，避免浮点误差。

## 最小实现

**本章接口：** `min_eating_speed(piles, h)`、`ship_within_days(weights, days)`、`first_true(lo, hi, feasible)`

In [1]:
def first_true(lo,hi,feasible):
    """Smallest feasible integer in inclusive [lo,hi]; hi must be feasible."""
    if lo>hi or not feasible(hi): raise ValueError('upper bound must be feasible')
    while lo<hi:
        mid=(lo+hi)//2
        if feasible(mid): hi=mid
        else: lo=mid+1
    return lo

def min_eating_speed(piles,h):
    if not piles or h<len(piles): raise ValueError('positive piles and enough hours required')
    return first_true(1,max(piles),lambda speed:sum((p+speed-1)//speed for p in piles)<=h)

def ship_within_days(weights,days):
    if not weights or days<1: raise ValueError('nonempty positive weights and positive days required')
    def feasible(capacity):
        used=1; load=0
        for w in weights:
            if load+w>capacity: used+=1; load=0
            load+=w
        return used<=days
    return first_true(max(weights),sum(weights),feasible)

## 正确性、前提与复杂度

吃香蕉判定可分离求和。装船判定中每一天尽量装到容量上限，不会让以后剩余前缀更多；对天数做“领先”归纳即可证明这是最少天数。单调可行集合的首个元素就是最优解。

**代价：** 每次判定 O(n)，二分次数 O(log U)，总 O(n log U)、O(1)辅助空间；U为答案搜索跨度。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

p=[3,6,7,11]
show_table(['速度','总小时','≤8可行'],[(v,sum((x+v-1)//v for x in p),sum((x+v-1)//v for x in p)<=8) for v in range(1,12)])

速度,总小时,≤8可行
1,27,False
2,15,False
3,10,False
4,8,True
5,8,True
6,6,True
7,5,True
8,5,True
9,5,True
10,5,True


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert min_eating_speed([3,6,7,11],8)==4
assert ship_within_days([1,2,3,4],1)==10
assert ship_within_days([1,2,3,4],4)==4
from itertools import product
for a in product([1,2,3],repeat=4):
    for h in range(4,9):
        ref=next(v for v in range(1,max(a)+1) if sum((x+v-1)//v for x in a)<=h)
        assert min_eating_speed(a,h)==ref
for boundary in range(21):
    assert first_true(0,20,lambda x:x>=boundary)==boundary
print('N043: 所有本章断言通过')

N043: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 为运力判定给出贪心装载论证。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造不可单调谓词说明不能二分。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 875. Koko Eating Bananas（canonical）
- 1011. Capacity To Ship Packages Within D Days（canonical）
- 410. Split Array Largest Sum（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。